In [1]:

import locale
locale.getpreferredencoding = lambda: "UTF-8"


In [2]:
!pip install cudf-cu12 cuml-cu12 --extra-index-url=https://pypi.nvidia.com


Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 78.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 206.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 917.0/917.0 kB 210.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 144.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 77.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 MB 141.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 851.7/851.7 kB 211.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 208.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of cuml-cu12 to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 197.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9

In [9]:
from google.cloud import storage
from google.cloud import bigquery

import os
import re
from tqdm import tqdm
import numpy as np
from collections import defaultdict
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
import sys
import pandas as pd
import gcsfs
import pyarrow

from google.cloud import bigquery


#client = storage.Client()
#bucket = client.bucket('cdow')


# Initialize the BigQuery client
client = bigquery.Client()

# Define the table reference, which was used in the previous cell
table_ref = "expanded-nebula-754.sandbox_crdow.leads_training_set_text_category"

# Use the BigQuery client to download the table directly into a DataFrame.
# This is more reliable than exporting to GCS and then reading from GCS.
print("Loading data directly from BigQuery...")
df = client.query(f"SELECT * FROM `{table_ref}`").to_dataframe()

print("Successfully loaded data into a DataFrame.")



Loading data directly from BigQuery...
Successfully loaded data into a DataFrame.


In [10]:
pd.set_option('display.max_rows', None)
#["a_campaign_name"]
#["a_leads_recent"]
# Show all columns
pd.set_option('display.max_columns', None)

# Prevent column width truncation
pd.set_option('display.max_colwidth', None)

# Optional: widen the display in Colab
from IPython.display import display
df_ = df.iloc[0:1500].copy()


display(df.loc[0:25, :])
print(df.dtypes)

opportunity_id                                 a_email  \
0   006dk000001aOmQAAU                   [bhbio1801@naver.com]   
1   006Hn00001RlnVnIAJ           [nickolas.perry.mil@usmc.mil]   
2   0061E00001LdLBJQA3                    [kaiyue.ma@yale.edu]   
3   0061E00001LLc1lQAD              [takeshi.oku@mystar.co.jp]   
4   006dk000004kgDIAAY                     [gms221@lehigh.edu]   
5   0061E00001LKBvBQAX             [chirag.patel7@utlmail.com]   
6   006Hn00001PzawQIAR                [beninga@impres-gmbh.de]   
7   006Hn00001MJSiGIAX          [14191802792@vip.hnist.edu.cn]   
8   006dk000004OkIkAAK                [maxime.vannier@hnfc.fr]   
9   006dk000002bhBJAAY              [natalie.haddad@emory.edu]   
10  006dk000006t2aPAAQ                   [wangmei@sxsb369.com]   
11  006Hn00001MJlzDIAT                     [2453006496@qq.com]   
12  006Hn00001Pxh19IAB        [pauline.fournit@merckgroup.com]   
13  006dk0000020gUMAAY       [cherie.webber@unchealth.unc.edu]   
14  006Hn00001RGKnrIAH                 [ahmad290884@gmail.com]   
15  006dk000006sYsTAAU            [rajaram.farakate@cipla.com]   
16  0061E00001KEIigQAH                     [ktekwani@udel.edu]   
17  0061E00001LfjM6QAJ               [chris.a.delre@gmail.com]   
18  006Hn00001RhebfIAB  [girishhalemirle-rajacharya@ouhsc.edu]   
19  0061E00001Ji2XLQAZ               [agtelang@rediffmail.com]   
20  006dk000008Fn3rAAC                 [victoria@winechek.com]   
21  0061E00001LfgyCQAR            [vlaffille@cristal-union.fr]   
22  0061E00001KDvOEQA1                       [xing.xu@fmc.com]   
23  006dk000004xeolAAA           [helixtechnology@hotmail.com]   
24  0061E00001LXCH7QAP               [sharmasn79.as@gmail.com]   
25  006dk000004LRKgAAO                   [cludy@somersetmd.us]   

                 a_search_terms                   a_source_campaigns  \
0                            []                                   []   
1                            []                                   []   
2                            []                                   []   
3                            []                                   []   
4                            []                                   []   
5                            []                                   []   
6                            []                                   []   
7                            []                                   []   
8                            []                                   []   
9                            []                                   []   
10                           []                                   []   
11                           []                                   []   
12                           []                                   []   
13                           []                                   []   
14                           []                                   []   
15                           []                                   []   
16                           []                                   []   
17                           []                                   []   
18                           []                                   []   
19                           []                                   []   
20  [L- 28 2B 29-Tartaric acid]  [all product_dsa_ww_(bing ebizpfs)]   
21                           []                                   []   
22                           []                                   []   
23                           []                                   []   
24                           []                                   []   
25                           []                                   []   

   a_currency  \
0       [KRW]   
1       [USD]   
2       [USD]   
3       [JPY]   
4       [USD]   
5       [INR]   
6       [EUR]   
7       [CNY]   
8       [EUR]   
9       [USD]   
10      [CNY]   
11      [CNY]   
12      [EUR]   
13      [USD]   
14      [IDR]   
15      

opportunity_id         object
a_email                object
a_search_terms         object
a_source_campaigns     object
a_currency             object
a_product_interest     object
a_leads_recent         object
a_url                  object
a_recordtypename_c     object
a_is_won               object
a_is_closed            object
a_Type                 object
a_campaign_name        object
page_views              Int64
days_open             float64
a_emailsubject         object
dtype: object


In [14]:
from typing import List, Dict, Optional, Union
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
import joblib
import math

class MultiColumnStringTokenEncoder:
    def __init__(
        self,
        columns: Union[List[str], Dict[str, int]],
        n_tokens: Optional[int] = 1000,
        svd_dim: int = 128,
        use_gpu: bool = False,
        batch_size: int = 20000,
        random_state: int = 42,
    ):
        if isinstance(columns, dict):
            self.columns = list(columns.keys())
            self._n_tokens_per_col = dict(columns)
        else:
            self.columns = list(columns)
            self._n_tokens_per_col = {c: n_tokens for c in self.columns}

        self.svd_dim = svd_dim
        self.use_gpu = use_gpu
        self.batch_size = batch_size
        self.random_state = random_state
        self.col_state = {}

        # Select backend
        if self.use_gpu:
            try:
                import cuml
                from cuml.feature_extraction.text import TfidfVectorizer
                from cuml.decomposition import TruncatedSVD
                from cuml.cluster import KMeans
                self.Vectorizer = TfidfVectorizer
                self.SVD = TruncatedSVD
                self.Cluster = KMeans
                self.backend = "gpu"
            except ImportError:
                print(" cuML not available, falling back to sklearn CPU")
                self._set_cpu_backend()
        else:
            self._set_cpu_backend()

    def _set_cpu_backend(self):
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.decomposition import TruncatedSVD
        from sklearn.cluster import MiniBatchKMeans
        self.Vectorizer = TfidfVectorizer
        self.SVD = TruncatedSVD
        self.Cluster = MiniBatchKMeans
        self.backend = "cpu"

    def _normalize_cell(self, cell):
        if cell is None or (isinstance(cell, float) and math.isnan(cell)):
            return []
        if isinstance(cell, str):
            return [cell] if cell.strip() else []
        if hasattr(cell, "__iter__"):
            out = []
            for s in cell:
                if s is None:
                    continue
                if isinstance(s, float) and math.isnan(s):
                    continue
                s = str(s)
                if s.strip():
                    out.append(s)
            return out
        return [str(cell)]

    def _gather_unique(self, df: pd.DataFrame, column: str):
        counter = Counter()
        for cell in df[column]:
            counter.update(self._normalize_cell(cell))
        return counter

    def fit_column(self, df: pd.DataFrame, column: str, n_tokens: int):
        freq = self._gather_unique(df, column)
        if not freq:
            self.col_state[column] = {
                "vectorizer": None,
                "svd": None,
                "centers": np.zeros((0, self.svd_dim), dtype=np.float32),
                "str2token": {},
                "token2repr": {},
                "tokens_count": 0,
            }
            return

        strings = list(freq.keys())
        counts = [freq[s] for s in strings]
        n_clusters = min(n_tokens, len(strings))

        vec = self.Vectorizer(
            analyzer="char",
            ngram_range=(1, 6),
            lowercase=False,
            dtype=np.float32,
        )
        print(f"[{self.backend}] fitting vectorizer on {len(strings)} strings")
        print(len(pd.Series(strings)))
        #print(pd.Series(counts).shape)
        X = vec.fit_transform(pd.Series(strings))
        print(type(X))

        svd_dim = min(self.svd_dim, X.shape[1] - 1) if X.shape[1] > 1 else 1
        svd = self.SVD(n_components=svd_dim, random_state=self.random_state)
        Xr = svd.fit_transform(X).astype(np.float32)

        km = self.Cluster(n_clusters=n_clusters, random_state=self.random_state)
        km.fit(Xr)
        centers = km.cluster_centers_.astype(np.float32)
        labels = km.predict(Xr)

        str2token = {}
        token2repr = {}
        members = defaultdict(list)
        for s, l in zip(strings, labels):
            t = int(l) + 1
            str2token[s] = t
            members[t].append(s)
        for t, group in members.items():
            best = max(group, key=lambda s: freq[s])
            token2repr[t] = best

        self.col_state[column] = {
            "vectorizer": vec,
            "svd": svd,
            "centers": centers,
            "str2token": str2token,
            "token2repr": token2repr,
            "tokens_count": n_clusters,
        }

    def fit(self, df: pd.DataFrame, verbose=False):
        for col in self.columns:
            n_tokens = self._n_tokens_per_col[col]
            if verbose:
                print(f"[{self.backend}] fitting {col} with {n_tokens} tokens")
            self.fit_column(df, col, n_tokens)

    def _nearest_token(self, vec, column: str):
        state = self.col_state[column]
        centers = state["centers"]
        if centers.shape[0] == 0:
            return 0
        v2 = np.sum(vec * vec, axis=1, keepdims=True)
        c2 = np.sum(centers * centers, axis=1)
        dist = v2 - 2 * vec.dot(centers.T) + c2
        return np.argmin(dist, axis=1) + 1

    def encode_strings(self, strings: List[str], column: str) -> List[int]:
        state = self.col_state[column]
        vec, svd, str2token = (
            state["vectorizer"],
            state["svd"],
            state["str2token"],
        )
        out = []
        unseen = []
        idxs = []
        for i, s in enumerate(strings):
            if not s or s.strip() == "":
                out.append(0)
                continue
            if s in str2token:
                out.append(str2token[s])
            else:
                out.append(None)
                unseen.append(s)
                idxs.append(i)

        # Batch transform unseen strings to avoid GPU OOM
        if unseen:
            for i0 in range(0, len(unseen), self.batch_size):
                batch = unseen[i0 : i0 + self.batch_size]
                X = vec.transform(batch)
                Xr = svd.transform(X).astype(np.float32)
                toks = self._nearest_token(Xr, column)
                for j, t in zip(idxs[i0 : i0 + self.batch_size], toks):
                    out[j] = int(t)
        return out

    def transform(self, df: pd.DataFrame, replace=False, suffix="_tok"):
        out = df.copy()
        for col in self.columns:
            tgt = col if replace else col + suffix
            out[tgt] = df[col].apply(
                lambda c: self.encode_strings(self._normalize_cell(c), col)
            )
        return out

    def decode_token(self, token: int, column: str) -> Optional[str]:
        if token == 0:
            return None
        return self.col_state[column]["token2repr"].get(token)

    def save(self, path): joblib.dump(self.col_state, path)
    def load(self, path): self.col_state = joblib.load(path)


In [ ]:
df = pd.DataFrame({
    "urls": [
        ["a.com/x", "a.com/y"],
        ["b.com"],
        [],
        ["c.com/path", "a.com/x"],
        [""], # empty string inside list
    ],
    "names": [
        ["Chris", "Christopher"],
        ["chris"],
        [],
        ["Kriss"],
        ["Bob"]
    ]
})
#print(df)


enc = MultiColumnStringTokenEncoder(
    {"urls": 3, "names": 3}, # max 3 tokens per column
    use_gpu=False, # CPU mode for demo
)
enc.fit(df, verbose=True)

encoded = enc.transform(df)
print(encoded)


[cpu] fitting urls with 3 tokens
[cpu] fitting vectorizer on 4 strings
4
<class 'scipy.sparse._csr.csr_matrix'>
[cpu] fitting names with 3 tokens
[cpu] fitting vectorizer on 5 strings
5
<class 'scipy.sparse._csr.csr_matrix'>
                    urls                 names urls_tok names_tok
0     [a.com/x, a.com/y]  [Chris, Christopher]   [2, 2]    [1, 1]
1                [b.com]               [chris]      [1]       [1]
2                     []                    []       []        []
3  [c.com/path, a.com/x]               [Kriss]   [3, 2]       [3]
4                     []                 [Bob]       []       [2]


In [36]:

enc = MultiColumnStringTokenEncoder(
    {"a_campaign_name": 500, "a_leads_recent": 20, "a_search_terms":4000}, # max 3 tokens per column
    use_gpu=False, # CPU mode for demo
)
enc.fit(df, verbose=True)

encoded = enc.transform(df)




[cpu] fitting a_campaign_name with 500 tokens
[cpu] fitting vectorizer on 6320 strings
6320
<class 'scipy.sparse._csr.csr_matrix'>
[cpu] fitting a_leads_recent with 20 tokens
[cpu] fitting vectorizer on 68 strings
68
<class 'scipy.sparse._csr.csr_matrix'>
[cpu] fitting a_search_terms with 4000 tokens
[cpu] fitting vectorizer on 15933 strings
15933
<class 'scipy.sparse._csr.csr_matrix'>


In [34]:
pd.set_option('display.max_rows', None)
#["a_campaign_name"]
#["a_leads_recent"]a_campaign_name_tok	a_leads_recent_tok
# Show all columns
pd.set_option('display.max_columns', None)

# Prevent column width truncation
pd.set_option('display.max_colwidth', None)

# Optional: widen the display in Colab
from IPython.display import display

display(encoded["a_campaign_name_tok"].max())
display(encoded["a_campaign_name_tok"].min())
display(encoded["a_leads_recent_tok"].max())
display(encoded["a_leads_recent_tok"].min())
display(encoded["a_search_terms_tok"].max())
display(encoded["a_search_terms_tok"].min())

[200]

[]

[20]

[]

[1000]

[]

In [37]:
pd.set_option('display.max_rows', None)
#["a_campaign_name"]
#["a_leads_recent"]a_campaign_name_tok	a_leads_recent_tok
# Show all columns
pd.set_option('display.max_columns', None)

# Prevent column width truncation
pd.set_option('display.max_colwidth', None)

# Optional: widen the display in Colab
from IPython.display import display
#df = df.iloc[0:1500].copy()


display(encoded.loc[200:570,["a_campaign_name","a_leads_recent","a_search_terms","a_campaign_name_tok","a_leads_recent_tok","a_search_terms_tok"]])

,a_campaign_name,a_leads_recent,a_search_terms,a_campaign_name_tok,a_leads_recent_tok,a_search_terms_tok
200,[2025-WE-LWH-LW Consistent water quality-Consumables],[],[],[288],[],[]
201,[2023-IN-AC-iFANS Conference-lead generation],[Parent Campaign],[],[90],[12],[]
202,[2025-IN-LWC-LW Consistent Water Quality-Consumables],[Referral],[],[154],[14],[]
203,[2025-IN-LWH-LW Consistent Water Quality-Leads from Distributor],[Referral],[],[154],[14],[]
204,[Bulk Web Form-SIAL-ID82],[Internet],[],[433],[2],[]
205,[2023-CN-90th anniversary in China-细胞福袋活动],[],[],[119],[],[]
206,[2024-CN-LWH-LW Consistent Water Quality-CN New SC leads],[],[],[172],[],[]
207,[NA Lead Share],[Sales Support],[],[123],[3],[]
208,[2023-TW-LS-LINE request],[],[],[347],[],[]
209,[NA Lead Share],[Referral],[change membrane millipore afs 8],[123],[14],[1132]


In [ ]:
from collections import Counter
from sklearn.base import BaseEstimator, TransformerMixin
import json


class QuantileCategoricalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols, max_len=40, missing_token="__MISSING__",
                 empty_token="__EMPTY__", rare_quantile=0.1):
        """
        cols: list of column names to encode
        max_len: maximum substring length per token
        missing_token: category for rare/unseen strings
        empty_token: category for empty lists
        rare_quantile: bottom quantile of frequency to bin into missing_token
        """
        self.cols = cols
        self.max_len = max_len
        self.missing_token = missing_token
        self.empty_token = empty_token
        self.rare_quantile = rare_quantile
        self.vocabs = {}
        self.counter = []

    def _normalize(self, arr):
        """Normalize list/ndarray/scalar to a truncated string token."""
        if arr is None:
            return self.empty_token

        if isinstance(arr, (list, np.ndarray)):
            if len(arr) == 0:
                return self.empty_token
            s = " ".join(str(x) for x in arr)
        else: # scalar string, number, etc.
            s = str(arr)

        return s[: self.max_len]

    def fit(self, X, y=None):
        """Build vocabularies based on frequency quantiles."""
        X = pd.DataFrame(X) # ensure dataframe
        for col in self.cols:
            # Count normalized strings
            self.counter = Counter(self._normalize(val) for val in X[col])

            freqs = np.array(list(self.counter.values()))
            threshold = np.quantile(freqs, self.rare_quantile)

            vocab = {}
            for token, count in self.counter.items():
                if count > threshold and token not in (self.empty_token, self.missing_token):
                    vocab[token] = len(vocab) + 1 # reserve 0 for padding if needed

            # Add special tokens
            vocab[self.empty_token] = len(vocab) + 1
            vocab[self.missing_token] = len(vocab) + 1

            self.vocabs[col] = vocab

        return self

    def transform(self, X):
        """Transform columns into integer IDs."""
        X = pd.DataFrame(X).copy()
        for col in self.cols:
            vocab = self.vocabs[col]
            miss_id = vocab[self.missing_token]
            empty_id = vocab[self.empty_token]

            def encode(val):
                s = self._normalize(val)
                if s == self.empty_token:
                    return empty_id
                return vocab.get(s, miss_id)

            X[col] = X[col].apply(encode)

        return X

    def inverse_transform(self, X):
        """Convert integer IDs back to strings (best effort)."""
        X = pd.DataFrame(X).copy()
        for col in self.cols:
            inv_vocab = {v: k for k, v in self.vocabs[col].items()}
            X[col] = X[col].apply(lambda i: inv_vocab.get(i, self.missing_token))
        return X

    def save(self, path):
        with open(path, "w") as f:
            json.dump(self.vocabs, f)

    def load(self, path):
        with open(path, "r") as f:
            self.vocabs = json.load(f)




In [ ]:
cat_col = ['a_search_terms','a_product_interest','a_campaign_name','a_Type','a_leads_recent','a_currency']

In [ ]:
encoder = QuantileCategoricalEncoder(cols=cat_col,max_len=20, rare_quantile=0.2)
df_encoded = encoder.fit_transform(df)

print("Encoded:\n", df_encoded.head())
print("Vocabs:\n", encoder.vocabs)


Encoded:
        opportunity_id                              a_email  a_search_terms  \
0  0061E00001LGs9tQAD              [lab@indoaminesltd.com]            4233   
1  0061E00001K3YmEQAV  [vijayvinayak.goregaonkar@mylan.in]            4233   
2  006Hn00001MYH9fIAH          [yasir.beeran@rudolfovo.eu]            4233   
3  006dk000006i2yoAAA      [sadettesalonvaara@eurofins.fi]            4234   
4  0061E00001LLE8TQAX       [maxime.perichard@univ-amu.fr]               1   

  a_source_campaigns  a_currency  a_product_interest  a_leads_recent  \
0                 []           1                 193             196   
1                 []           1                 193             196   
2                 []           2                 193             196   
3       [(referral)]           2                   1               1   
4       [(referral)]           2                   2               2   

                                                                                        

In [ ]:
pd.set_option('display.max_rows', None)

# Show all columns
pd.set_option('display.max_columns', None)

# Prevent column width truncation
pd.set_option('display.max_colwidth', None)

# Optional: widen the display in Colab
from IPython.display import display
display(encoder.vocabs)

{'a_search_terms': {'Silica plates': 1,
  '345245-100ML': 2,
  '366927': 3,
  '62478 1.02428': 4,
  'fibronectin': 5,
  'millipore elix advan': 6,
  '592-48-3': 7,
  'Ethyl Alcohol 2C 200': 8,
  '1057440001': 9,
  'iron nitrate': 10,
  'particle counters': 11,
  'merk': 12,
  '8018049030 benzoyl c': 13,
  'sonicator': 14,
  'ethyl acetate': 15,
  '109098': 16,
  'dry-ice-storage-box': 17,
  'aquafine 52885 ds15z': 18,
  '6484-52-2': 19,
  'G7645': 20,
  'nad 2B': 21,
  'D8537': 22,
  'dmem': 23,
  'EDTA': 24,
  'hagp1mag-12k': 25,
  '13-676-61F': 26,
  'e0282': 27,
  'silicone oil for oil': 28,
  'L2524-10KU 260913-25': 29,
  '481973': 30,
  '60357-50G 100-41-4': 31,
  'santa cruz antibodie': 32,
  'sodium dihydrogen ph': 33,
  'T1775': 34,
  '1.10020.0001': 35,
  'doxycycline': 36,
  '296309': 37,
  'camptothecin': 38,
  'certificate of analy': 39,
  'c102180': 40,
  'Contr C3 B4les posit': 41,
  '3-methoxy-4-nitrofla': 42,
  'pipet aid': 43,
  'cholesterol': 44,
  'l-?-phosphatidylch